# S1 · Range-aware evaluation harness

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/sct-range/blob/main/notebooks/s1_harness.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hashirama21/sct-range/blob/main/notebooks/s1_harness.ipynb)

This notebook validates every week-1 component and produces the artefacts that later weeks depend on:

1. a reproducible environment record;
2. a frozen, hash-verified local test split;
3. an HU → SPR conversion identical to the one used by the SynthRAD2025 dose engine (pyRadPlan), checked on the HU of a
   real CT;
4. a differentiable water-equivalent-thickness (WET) projector, checked against pyRadPlan's Siddon ray tracer and
   finite differences on real patient anatomy, and against exact chord lengths on disks;
5. per-case range diagnostics on the development pool.

All logic lives in the `sctrange` package; the notebook configures, runs and reports. Every check uses the real
data except the two analytic geometry checks (disk chords, target segments), which need a shape whose exact
path lengths are known; the same projector is then compared with the Siddon tracer on a patient.
The notebook runs on the **real SynthRAD2025 training data** (Zenodo record 15373853), read from its
Hugging Face mirror [developper669/synthrad2025](https://huggingface.co/datasets/developper669/synthrad2025)
(tag `v2`). `SYNTHRAD2025_ROOT` points to the local copy (default `data/synthrad2025` in the repository,
ignored by git); missing tasks are fetched first (`sctrange.download`, parallel and resumable, only the
requested task; `SCT_DOWNLOAD=0` forbids it and stops with an error instead). `SCT_TASKS` selects the tasks (default `1,2`).
`SCT_PREDICTION_ROOT` optionally points to a folder of `<case_id>.mha` predictions to diagnose next to the
bulk-density baseline; `SCT_MAX_CASES` limits the diagnostics to the first cases (they take a few seconds per case on
a CPU). pyRadPlan is optional: without it the two dose-engine comparisons are reported as not run.

### Setup · install `sctrange` from the repository

Run this cell first. Inside a clone of [hashirama21/sct-range](https://github.com/hashirama21/sct-range) it installs that
clone in editable mode (`pip install -e ".[dose]"`), so the notebook always runs the repository's code, local edits
included. Opened on its own, from the Colab or Kaggle badge or on another machine, it first downloads the repository
from GitHub (no git needed) to `/content/sct-range` on Colab, `/kaggle/tmp/sct-range` on Kaggle (so that
`/kaggle/working` only receives outputs) or `./sct-range` elsewhere, then installs it. On Kaggle, enable
*Settings → Internet* first. `SCTRANGE_REF` selects a branch, tag or commit (default `main`), `SCTRANGE_HOME` the
destination; pip only runs when `sctrange` is missing or installed from elsewhere (`SCTRANGE_REINSTALL=1` forces it).

In [ ]:
import importlib
import importlib.util
import io
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

# OpenBLAS 0.3.27 can corrupt multithreaded matrix products, and pyRadPlan (S6) optimises plans and computes dose with
# NumPy: one BLAS thread per process. The variable only acts before NumPy is loaded; threadpoolctl covers a later cell.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
if "numpy" in sys.modules:
    try:
        from threadpoolctl import threadpool_limits

        BLAS_LIMIT = threadpool_limits(1, user_api="blas")
    except ImportError:
        print("NumPy was loaded before this cell: restart the kernel so that OPENBLAS_NUM_THREADS=1 applies")

REPO_URL = "https://github.com/hashirama21/sct-range"
REPO_REF = os.environ.get("SCTRANGE_REF", "main")
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir()


def find_repo(start: Path) -> Path | None:
    return next((p for p in (start, *start.parents) if (p / "pyproject.toml").exists() and (p / "src" / "sctrange").is_dir()),
                None)


def download_repo(destination: Path) -> Path:
    """Fetch ``REPO_REF`` of the GitHub repository as a zip archive and unpack it to ``destination``."""
    for attempt in range(4):
        try:
            with urllib.request.urlopen(f"{REPO_URL}/archive/{REPO_REF}.zip", timeout=120) as response:
                archive = zipfile.ZipFile(io.BytesIO(response.read()))
            break
        except OSError as error:
            if attempt == 3:
                raise
            print(f"download failed ({error}); retrying")
            time.sleep(5 * (attempt + 1))
    top = archive.namelist()[0].split("/")[0]
    staging = destination.with_name(destination.name + ".partial")
    shutil.rmtree(staging, ignore_errors=True)
    archive.extractall(staging)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():  # an earlier, interrupted attempt or another folder
        if any(destination.iterdir()):
            raise RuntimeError(f"{destination} exists and is not a clone of {REPO_URL}; remove it or set SCTRANGE_HOME")
        destination.rmdir()
    (staging / top).rename(destination)
    shutil.rmtree(staging, ignore_errors=True)
    return destination


REPO_ROOT = find_repo(Path.cwd())
if REPO_ROOT is None:
    default_home = (Path("/content/sct-range") if ON_COLAB else Path("/kaggle/tmp/sct-range") if ON_KAGGLE
                    else Path.cwd() / "sct-range")
    REPO_ROOT = Path(os.environ.get("SCTRANGE_HOME", default_home)).resolve()
    if find_repo(REPO_ROOT) is None:
        print(f"downloading {REPO_URL} @ {REPO_REF} to {REPO_ROOT}")
        download_repo(REPO_ROOT)
    os.chdir(REPO_ROOT / "notebooks")
SOURCE_DIR = (REPO_ROOT / "src").resolve()


def installed_from_repo() -> bool:
    spec = importlib.util.find_spec("sctrange")
    return spec is not None and spec.origin is not None and Path(spec.origin).resolve().is_relative_to(SOURCE_DIR)


if os.environ.get("SCTRANGE_REINSTALL") == "1" or not installed_from_repo():
    command = [sys.executable, "-m", "pip", "install", "--quiet", "--editable", f"{REPO_ROOT}[dose]"]
    for attempt in range(1, 4):
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode == 0:
            break
        print(f"pip install failed (attempt {attempt}/3):\n{result.stderr[-3000:]}")
        time.sleep(10 * attempt)
    else:
        raise RuntimeError("pip could not install sctrange; see the messages above (network? Kaggle: enable Internet)")
# an editable install is only picked up by new processes; make it visible to this kernel too
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
for module in [name for name in sys.modules if name == "sctrange" or name.startswith("sctrange.")]:
    del sys.modules[module]
importlib.invalidate_caches()

import sctrange

assert Path(sctrange.__file__).resolve().is_relative_to(SOURCE_DIR), f"sctrange imported from {sctrange.__file__}"
print(f"sctrange {sctrange.__version__} installed from {REPO_ROOT.name}/src (editable)")

### Parameters

Set them here and run all cells (*Runtime → Run all* on Colab, where this cell is a form). Environment variables
of the same meaning, given in brackets, override them for unattended runs (`jupyter nbconvert --execute`).

The first run on the real data, with both tasks, freezes the local test split; commit it afterwards (on Colab a
copy is kept in Drive and used by S3 and S4).
| Platform | Dataset, cache | Outputs | GPU |
|---|---|---|---|
| local | `data/synthrad2025`, `outputs/` | `outputs/` | every visible GPU |
| Colab | `/content/data`, `/content/cache` | Google Drive `MyDrive/sct-range/outputs` (`USE_DRIVE`) | *Runtime → Change runtime type → T4 GPU* |
| Kaggle | `/kaggle/tmp/data`, `/kaggle/tmp/cache` | `/kaggle/working/outputs` | *Settings → Accelerator → GPU T4 × 2*, *Internet on* |

With two GPUs (Kaggle T4 × 2) the remaining folds train in parallel, one per GPU. T4 GPUs train in float16 with loss
scaling (they have no native bfloat16). Predictions are stored as int16 HU to fit the 20 GB of `/kaggle/working`.

In [ ]:
# @title Parameters { display-mode: "form" }
TASKS = "1,2"  # @param ["1,2", "1", "2"]  [SCT_TASKS]
DOWNLOAD = True  # @param {type:"boolean"}  fetch missing data from Hugging Face [SCT_DOWNLOAD]
MAX_CASES = 0  # @param {type:"integer"}  0 = every development case [SCT_MAX_CASES]
USE_DRIVE = True  # @param {type:"boolean"}  Colab only

In [ ]:
import importlib.util
import json
import os
import platform
import subprocess
import warnings
from importlib import metadata

warnings.filterwarnings("ignore", message=r"`torch\.jit\.script` is deprecated")
warnings.filterwarnings("ignore", message="IProgress not found")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
from IPython.display import display

from sctrange.data import bulk_density_sct, central_slab, discover_cases, load_case, read_prediction, read_volume
from sctrange.download import ensure_dataset, validate_dataset
from sctrange.runs import cached_result, config_key, files_signature
from sctrange.evaluation import DiagnosticConfig, evaluate_case, prepare_case
from sctrange.hlut import HLUT
from sctrange.metrics import range_error
from sctrange.training import visible_devices
from sctrange.runtime import frozen_split_path, resolve_paths, save_split_copy
from sctrange.plotting import apply_style, plot_case, plot_hlut, plot_mae_vs_range, plot_projector_checks
from sctrange.split import SplitSpec, load_split, select_cases, split_hash, stratified_split, write_split
from sctrange.validation import (
    benchmark_range_loss,
    disk_chord_errors,
    gradcheck_range_loss,
    siddon_relative_errors,
    siddon_relative_errors_case,
    target_path_errors,
)

PYRADPLAN = importlib.util.find_spec("pyRadPlan") is not None

In [ ]:
PATHS = resolve_paths(REPO_ROOT, USE_DRIVE)
TASKS = tuple(os.environ.get("SCT_TASKS", TASKS).split(","))
DOWNLOAD = os.environ.get("SCT_DOWNLOAD", "1" if DOWNLOAD else "0") != "0"
MAX_CASES = int(os.environ.get("SCT_MAX_CASES", MAX_CASES)) or None
DATA_ROOT = PATHS.data_root
PREDICTION_ROOT = Path(os.environ["SCT_PREDICTION_ROOT"]) if "SCT_PREDICTION_ROOT" in os.environ else None
OUTPUT_DIR = PATHS.output_root / "s1"
FIGURE_DIR = OUTPUT_DIR / "figures"
FROZEN_SPLIT_PATH = frozen_split_path(REPO_ROOT, PATHS)

SEED = 20261013
TEST_FRACTION = 0.2
DEVICES = visible_devices()  # ["cuda:0", "cuda:1"] on Kaggle T4 x 2
DEVICE = DEVICES[0]
VALIDATION_ANGLES = (0.0, 17.0, 45.0, 90.0, 133.0, 210.0, 301.0)
SIDDON_ANGLES = (0.0, 60.0, 135.0)
DIAGNOSTIC = DiagnosticConfig(angles_deg=tuple(range(0, 360, 45)), target_radius_mm=25.0, seed=SEED, device=DEVICE)

torch.manual_seed(SEED)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
apply_style()
display(pd.Series(PATHS.describe(REPO_ROOT), name="location").to_frame())

In [ ]:
DATA_ROOT = ensure_dataset(DATA_ROOT, TASKS, download=DOWNLOAD)
SPLIT_PATH = FROZEN_SPLIT_PATH


def shown(path: Path | None) -> str:
    """Repository-relative path when possible, so that outputs carry no machine-specific prefix."""
    if path is None:
        return "none"
    try:
        return path.resolve().relative_to(REPO_ROOT.resolve()).as_posix()
    except ValueError:
        return str(path)


print(f"data       : {shown(DATA_ROOT)} (tasks {', '.join(TASKS)})")
print(f"predictions: {shown(PREDICTION_ROOT)}")
print(f"split      : {shown(SPLIT_PATH)}")
print(f"device     : {DEVICE}")
print(f"pyRadPlan  : {'installed' if PYRADPLAN else 'not installed (engine comparisons skipped)'}")

## 0 · Dataset integrity

Every case is checked before use: `ct`, `mask` and the source image present, identical geometry, isotropic in-plane
spacing, a non-empty body mask and CT values inside [-1024, 3071] HU (the clipping of the v2 release). On the real
dataset the result comes with the data in `<root>/manifest.csv` (published with the Hugging Face
mirror); delete it to re-check locally.

In [ ]:
manifest_path = DATA_ROOT / "manifest.csv"
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path, dtype={"task": str, "case_id": str}, keep_default_na=False)
    manifest["ok"] = manifest["ok"].astype(str).str.lower() == "true"
else:
    manifest = validate_dataset(DATA_ROOT, TASKS)
manifest = manifest[manifest["task"].isin(TASKS)]
INVALID_CASES = set(manifest.loc[~manifest["ok"], "case_id"])

display(manifest.groupby(["task", "region", "center"])["ok"].agg(cases="size", valid="sum"))
if INVALID_CASES:
    display(manifest.loc[~manifest["ok"], ["case_id", "problems"]])
print(f"{int(manifest['ok'].sum())}/{len(manifest)} cases pass; invalid cases are left out of the diagnostics")

## 1 · Frozen local test split

SynthRAD2025 keeps its test ground truth private, so model selection needs a local hold-out drawn from the public training cases. The split is stratified by task, region and centre and uses an independent seed per stratum, so adding cases to one stratum never moves cases in another. It is written once with its SHA-256, committed, and only verified afterwards: `load_split` refuses a modified file.

On the real dataset the canonical file is `splits/synthrad2025_local_test.json`. It can also be produced from the command line and committed:

```bash
sctrange-split --data-root "$SYNTHRAD2025_ROOT" --out splits/synthrad2025_local_test.json
git add splits/ && git commit -m "Freeze SynthRAD2025 local test split"
```

The first run on the real data creates the file; commit it then (`splits/synthrad2025_local_test.json` and its
`.sha256`). Every later run, and every later notebook, only verifies it.

The first valid development case is the *reference case* of the checks on real anatomy below; test cases are
never loaded.

In [ ]:
records = [r for r in discover_cases(DATA_ROOT) if r.task in TASKS]
available = {record.case_id for record in records}

if SPLIT_PATH.exists():
    split = load_split(SPLIT_PATH)
    split_status = "verified existing split"
else:
    if set(TASKS) != {"1", "2"}:
        raise RuntimeError("the frozen split is created once over both tasks: run with TASKS = \"1,2\" the first time")
    split = stratified_split(records, SplitSpec(test_fraction=TEST_FRACTION, seed=SEED))
    write_split(split, SPLIT_PATH)
    split = load_split(SPLIT_PATH)
    split_status = "created and verified"
    print(f"frozen split created: commit {SPLIT_PATH.name} and its .sha256 (git add splits/)")
copy = save_split_copy(SPLIT_PATH, PATHS)
if copy is not None:
    print(f"copy kept with the outputs: {copy}")

assigned = set(split["test"]) | set(split["development"])
split_summary = {
    "status": split_status,
    "sha256": split_hash(split),
    "test cases": len(split["test"]),
    "development cases": len(split["development"]),
    "cases on disk not in split": len(available - assigned),
    "split cases missing on disk": len({c for c in assigned if c[0] in TASKS} - available),
}
assert not set(split["test"]) & set(split["development"])

display(pd.Series(split_summary, name="value").to_frame())
display(pd.DataFrame(split["strata"]).T.rename_axis("stratum"))

In [ ]:
development = [record for record in records
               if record.case_id in set(split["development"]) and record.case_id not in INVALID_CASES]
if MAX_CASES:  # a region-stratified subset, not the first case ids (which all come from one region and centre)
    keep = set(select_cases(development, MAX_CASES, SEED))
    development = [record for record in development if record.case_id in keep]
if not development:
    raise RuntimeError(f"no valid development case of task(s) {', '.join(TASKS)} under {DATA_ROOT}")
REFERENCE = development[0]
volumes = load_case(REFERENCE, ("ct", "mask"))
REF_CT, REF_MASK, REF_SPACING = volumes["ct"].array, volumes["mask"].array, volumes["ct"].spacing
print(f"reference case: {REFERENCE.case_id} ({REFERENCE.region}, centre {REFERENCE.center}), "
      f"{REF_CT.shape[0]} slices of {REF_CT.shape[1]} × {REF_CT.shape[2]}, spacing {tuple(round(s, 2) for s in REF_SPACING)} mm")

## 2 · Environment

Versions and the code revision are stored with every result so that any number can be traced back to the exact software that produced it.

In [ ]:
def package_version(name: str) -> str:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "not installed"


def git_revision(root: Path) -> str:
    try:
        revision = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=root, capture_output=True,
                                  text=True, check=True).stdout.strip()
        dirty = subprocess.run(["git", "status", "--porcelain"], cwd=root, capture_output=True, text=True).stdout
        return revision + ("-dirty" if dirty.strip() else "")
    except (OSError, subprocess.CalledProcessError):
        return "no git repository"


environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": DEVICE,
    **{name: package_version(name) for name in ("numpy", "scipy", "SimpleITK", "pandas", "pyradplan", "sctrange")},
    "git_revision": git_revision(REPO_ROOT),
    "dataset": f"SynthRAD2025 training v2, tasks {', '.join(TASKS)}",
}
(OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
display(pd.Series(environment, name="value").to_frame())

## 3 · HU → SPR conversion

The range proxy must convert HU exactly as the dose engine does. The SynthRAD2025 dose evaluation uses pyRadPlan (v0.2.8), whose default table is identical in v0.5.0 and is applied with `np.interp`: linear interpolation, constant beyond the end nodes.

Checks: the bundled table equals the installed one; the NumPy and Torch paths reproduce `CT.compute_wet` on the HU of four slices of the reference CT; the curve is monotone with water at 1.

This is the engine's generic table, not a scanner-specific stoichiometric calibration. It is the right choice for consistency with the challenge dosimetry; if a calibrated curve is adopted, it must replace the table in both the proxy and the dose engine.

In [ ]:
hlut = HLUT.pyradplan_default()
hlut_checks = {
    "fingerprint": hlut.fingerprint(),
    "monotone": bool(np.all(np.diff(hlut.numpy(np.linspace(-2000, 4000, 6001))) >= 0)),
    "SPR at 0 HU": float(hlut.numpy(np.array(0.0))),
}
if PYRADPLAN:
    from pyRadPlan.ct import CT

    installed = HLUT.from_pyradplan()
    probe = np.ascontiguousarray(REF_CT[central_slab(REF_MASK, 4)], dtype=np.float32)
    image = sitk.GetImageFromArray(probe)
    image.SetSpacing(tuple(float(s) for s in REF_SPACING))
    engine = sitk.GetArrayFromImage(CT(cube_hu=image).compute_wet(np.stack([hlut.hu, hlut.spr], axis=1)))
    hlut_checks.update({
        "matches installed pyRadPlan": bool(np.array_equal(installed.hu, hlut.hu) and np.array_equal(installed.spr, hlut.spr)),
        "max |numpy - engine|": float(np.abs(hlut.numpy(probe) - engine).max()),
        "max |torch - engine|": float(np.abs(hlut.torch(torch.from_numpy(probe).double()).numpy() - engine).max()),
        "HU values compared": f"{probe.size} ({REFERENCE.case_id}, {probe.min():.0f} to {probe.max():.0f} HU)",
    })
    assert hlut_checks["matches installed pyRadPlan"]
    assert hlut_checks["max |numpy - engine|"] < 1e-12 and hlut_checks["max |torch - engine|"] < 1e-12
else:
    print("pyRadPlan not installed: the bundled table is used without the engine comparison (pip install -e '.[dose]')")
assert hlut_checks["monotone"] and hlut_checks["SPR at 0 HU"] == 1.0

display(pd.Series(hlut_checks, name="value").to_frame())
display(pd.DataFrame({"HU": hlut.hu, "SPR": hlut.spr, "slope to next node (1/HU)": np.append(hlut.slopes(), np.nan)}))
plot_hlut(hlut).savefig(FIGURE_DIR / "hlut.png", bbox_inches="tight")
plt.show()

## 4 · Differentiable WET projector

Each axial slice is padded to a square canvas and rotated with `grid_sample` so that the beam travels along the row axis; a masked sum over rows gives one integral per ray, in millimetres of water. Segments run from beam entry either to the body exit (*chord*) or to the distal edge of the target (*target*), the latter being what governs the range of a proton stopping in the target.

Five independent checks, the last two on the reference patient:

| Check | Reference | Criterion |
|---|---|---|
| Disk chords | exact chord length `2√(R² − u²)` | median < 0.1 mm, max < 0.5 mm |
| Target segments | `√(R² − u²) + √(r² − u²)` | max < 1.5 mm (one pixel at the distal edge) |
| Heterogeneous phantom | pyRadPlan Siddon tracer (water, lung and bone inserts) | median < 1 %, max < 4 % |
| Real anatomy | pyRadPlan Siddon tracer on the largest slice of the reference CT, rays crossing ≥ 20 mm WET | median < 1 % |
| Gradient | finite differences in float64 on a 17 × 19 patch of the reference CT | `torch.autograd.gradcheck` passes |

In [ ]:
chord = disk_chord_errors(VALIDATION_ANGLES)
target = target_path_errors(VALIDATION_ANGLES)
empty = pd.DataFrame(columns=["angle_deg", "offset_px", "siddon_mm", "projector_mm", "relative_error"], dtype=float)
siddon = siddon_relative_errors(hlut, SIDDON_ANGLES) if PYRADPLAN else empty
siddon_real = (siddon_relative_errors_case(hlut, REF_CT, REF_MASK, REF_SPACING[0], SIDDON_ANGLES) if PYRADPLAN
               else empty)

projector_checks = {
    "chord median |error| (mm)": chord["median_abs_mm"].max(),
    "chord max |error| (mm)": chord["max_abs_mm"].max(),
    "target max |error| (mm)": target["max_abs_mm"].max(),
    "target rays valid inside / none outside": bool(target["valid_inside"].all() and not target["valid_outside"].any()),
    "Siddon median relative error": siddon["relative_error"].median() if len(siddon) else float("nan"),
    "Siddon max relative error": siddon["relative_error"].max() if len(siddon) else float("nan"),
    "Siddon rays compared": len(siddon),
    "real anatomy Siddon median relative error": siddon_real["relative_error"].median() if len(siddon_real) else float("nan"),
    "real anatomy Siddon p95 relative error": siddon_real["relative_error"].quantile(0.95) if len(siddon_real) else float("nan"),
    "real anatomy Siddon rays compared": len(siddon_real),
    "gradcheck": gradcheck_range_loss(hlut, SEED, REF_CT, REF_MASK),
}
assert projector_checks["chord median |error| (mm)"] < 0.1 and projector_checks["chord max |error| (mm)"] < 0.5
assert projector_checks["target max |error| (mm)"] < 1.5 and projector_checks["target rays valid inside / none outside"]
if PYRADPLAN:
    assert projector_checks["Siddon median relative error"] < 0.01 and projector_checks["Siddon max relative error"] < 0.04
assert projector_checks["gradcheck"]

display(pd.Series(projector_checks, name="value").to_frame())
display(chord.merge(target, on="angle_deg", suffixes=(" chord", " target")).round(3))
if len(siddon_real):
    display(siddon_real.groupby("angle_deg")[["siddon_mm", "relative_error"]]
            .agg(["median", "max"]).round(4).rename_axis(f"real anatomy, {REFERENCE.case_id}"))
plot_projector_checks(chord, target, siddon).savefig(FIGURE_DIR / "projector_checks.png", bbox_inches="tight")
plt.show()

Cost of one range-loss forward and backward pass on the reference case (16 slices around its largest body section, cropped to the body as in training, three beams), with the bulk-density baseline as sCT, on the active device.

In [ ]:
benchmark = benchmark_range_loss(hlut, REF_CT, REF_MASK, bulk_density_sct(REF_MASK), REF_SPACING[0],
                                 (0.0, 120.0, 240.0), device=DEVICE)
display(pd.Series({"case": REFERENCE.case_id, **benchmark}, name="value").to_frame())

## 5 · Range diagnostics on the development pool

Test cases are never loaded here. Each result is cached in `outputs/s1/cache_eval/`, so a rerun after an interrupted session only evaluates the cases that are missing. For each development case, the bulk-density baseline (water inside the body, as in the challenge) and, when available, a prediction are compared with the reference CT. SynthRAD2025 releases no PTV, so a 25 mm sphere at the body centroid serves as a surrogate target defining the distal edge.

Per case and source:

- `mae_hu`: mean absolute HU error in the body;
- `*_rms_mm`, `*_p95_mm`, `*_cvar_mm`: RMS, 95th percentile and upper-decile mean of |ΔWET| over rays, for chords and target segments;
- `*_coherence`: median of |Σε| / Σ|ε| along rays, close to 1/√n for white errors and to 1 for errors that accumulate;
- `*_intra_ratio`, `*_global_ratio`: RMS ΔWET divided by that of the same errors shuffled within tissue classes, or globally. Both shuffles leave the MAE unchanged, so a ratio away from 1 shows that range error depends on the spatial layout of errors and not only on their magnitude.

In [ ]:
prediction_label = "prediction"



def diagnose(record, path):
    volumes = load_case(record, ("ct", "mask"))
    ct, mask, spacing = volumes["ct"].array, volumes["mask"].array, volumes["ct"].spacing
    sct = bulk_density_sct(mask) if path is None else read_prediction(path, ct.shape)
    return evaluate_case(ct, mask, sct, spacing, hlut, DIAGNOSTIC)


rows = []
for index, record in enumerate(development, 1):
    paths = {"bulk density": None}
    if PREDICTION_ROOT is not None and (PREDICTION_ROOT / f"{record.case_id}.mha").exists():
        paths[prediction_label] = PREDICTION_ROOT / f"{record.case_id}.mha"
    data = files_signature([record.path("ct"), record.path("mask")], mtime=False)
    for source, path in paths.items():
        key = {"config": config_key(DIAGNOSTIC), "hlut": hlut.fingerprint(), "data": data,
               "prediction": files_signature([path]) if path else None}
        metrics = cached_result(OUTPUT_DIR / "cache_eval" / f"{record.case_id}_{source.replace(' ', '_')}.pkl", key,
                                lambda: diagnose(record, path))
        rows.append({"case_id": record.case_id, "region": record.region, "center": record.center,
                     "source": source, **metrics})
    if index % 50 == 0:
        print(f"{index}/{len(development)} cases", flush=True)

diagnostics = pd.DataFrame(rows)
diagnostics.to_csv(OUTPUT_DIR / "range_diagnostics.csv", index=False)
print(f"{diagnostics['case_id'].nunique()} development cases, {len(diagnostics)} evaluations")

columns = ["mae_hu", "chord_rms_mm", "target_rms_mm", "target_cvar_mm", "target_coherence",
           "target_intra_ratio", "target_global_ratio"]
display(diagnostics.groupby(["source", "region"])[columns].mean().round(3))

In [ ]:
plot_mae_vs_range(diagnostics).savefig(FIGURE_DIR / "mae_vs_range.png", bbox_inches="tight")
plt.show()
display(diagnostics.groupby("source")[["mae_hu", "target_rms_mm", "target_coherence"]]
        .corr(method="spearman").round(3))

One development case in detail: reference slice, signed HU error, and signed ΔWET per ray to the distal target edge for the first three beams. The slice shown is the one with the most target rays.

In [ ]:
example = next((row for row in rows if row["source"] == prediction_label), rows[0])
record = next(r for r in development if r.case_id == example["case_id"])
volumes = load_case(record, ("ct", "mask"))
sct = (read_volume(PREDICTION_ROOT / f"{record.case_id}.mha").array if example["source"] == prediction_label
       else bulk_density_sct(volumes["mask"].array))
case = prepare_case(volumes["ct"].array, volumes["mask"].array, sct, volumes["ct"].spacing, DIAGNOSTIC)
result = range_error(case.sct, case.ct, case.mask, case.projector, hlut, case.target)
slice_index = int(result.valid.sum(dim=(0, 2)).argmax())

plot_case(case.ct.cpu().numpy(), case.sct.cpu().numpy(), case.mask.cpu().numpy(),
          result.delta_mm.cpu().numpy(), result.valid.cpu().numpy(), DIAGNOSTIC.angles_deg,
          slice_index, f"{record.case_id} · {example['source']}").savefig(FIGURE_DIR / "example_case.png", bbox_inches="tight")
plt.show()

## 6 · S1 status

Every item is evaluated from the results above and written to `outputs/s1/s1_report.json` with the environment record.

In [ ]:
status = pd.DataFrame([
    ("pyRadPlan installed", environment["pyradplan"], environment["pyradplan"] != "not installed"),
    ("HU → SPR identical to dose engine", f"fingerprint {hlut.fingerprint()}",
     PYRADPLAN and hlut_checks["max |numpy - engine|"] < 1e-12 and hlut_checks["max |torch - engine|"] < 1e-12),
    ("Projector vs analytic chords", f"max {projector_checks['chord max |error| (mm)']:.3f} mm",
     projector_checks["chord max |error| (mm)"] < 0.5),
    ("Target segments to distal edge", f"max {projector_checks['target max |error| (mm)']:.3f} mm",
     projector_checks["target max |error| (mm)"] < 1.5),
    ("Projector vs pyRadPlan Siddon",
     f"median {projector_checks['Siddon median relative error']:.2%}" if PYRADPLAN else "not run (pyRadPlan missing)",
     PYRADPLAN and projector_checks["Siddon median relative error"] < 0.01),
    ("Dataset integrity", f"{int(manifest['ok'].sum())}/{len(manifest)} cases valid", not INVALID_CASES),
    ("Projector vs Siddon on real anatomy",
     f"median {projector_checks['real anatomy Siddon median relative error']:.2%} · {REFERENCE.case_id}"
     if PYRADPLAN else "not run (pyRadPlan missing)",
     PYRADPLAN and projector_checks["real anatomy Siddon median relative error"] < 0.01),
    ("Gradient check", f"float64 finite differences · {REFERENCE.case_id}", projector_checks["gradcheck"]),
    ("Local test split", f"{split_summary['status']} · sha256 {split_summary['sha256'][:12]}",
     split_summary["split cases missing on disk"] == 0),
    ("Split covers both tasks", shown(SPLIT_PATH),
     {"task1", "task2"} <= {stratum.split("/")[0] for stratum in split["strata"]}),
    ("Range diagnostics", f"{len(diagnostics)} evaluations", len(diagnostics) > 0),
], columns=["item", "detail", "passed"])

report = {"environment": environment, "hlut": hlut_checks, "projector": projector_checks,
          "benchmark": benchmark, "split": split_summary, "reference_case": REFERENCE.case_id,
          "invalid_cases": sorted(INVALID_CASES),
          "status": status.to_dict(orient="records")}
(OUTPUT_DIR / "s1_report.json").write_text(json.dumps(report, indent=2, default=str))
display(status)